
# Calibration run: 모든 arm 을 같은 각도로 두고 측정

positioner 들을 같은 각도(예: alpha −30°, beta 150°)나 회전 중심에서 같은 offset 으로 보낸 뒤 찍은 이미지로, 각 positioner 가 실제로 어디에 있는지와
목표까지의 회전량을 구한다. 목표 파일은 각도로 만들며(`lab.make_target`), `object.info` 자리에 그대로 쓸 수 있다
(`load_configuration` 이 각도도 읽는다). 나머지 흐름은 `Metrology_Debug.ipynb` 와 같다.


In [ ]:

import os, sys, json, glob
PKG_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))   # 이 노트북이 kspec_metrology/notebooks/ 에 있을 때 저장소 루트
OUT_DIR  = os.path.expanduser('~/metrology_out')                   # <- 결과를 쓸 폴더

# ---- 설정: 바꾸면 커널을 재시작한다 (표와 제외 목록은 import 때 읽힌다) ----
FIBER_TABLE   = None            # None = 패키지 기본 표 (config73). 다른 표면 경로
BROKEN_FIBERS = []              # 끊어져 빛이 안 나오는 fiber 의 홀 ID. 예: ['G4']
FIXED_FIBERS  = []              # 빛은 나오지만 고장나 움직이지 않는 positioner 의 홀 ID. 예: ['C6']
SKY_FIBERS    = ['A6', 'B7', 'B8', 'G0', 'H1', 'I0', 'N7', 'N8', 'O6']   # sky fiber 홀 (켜져 있으면). 꺼져 있으면 []
ANGLE_SIGN    = (1, 1)          # 출력 각도 = 부호 x 계산각 (alpha 반시계 양, beta >= 0)

if FIBER_TABLE:
    os.environ['KSPEC_FIBER_TABLE'] = FIBER_TABLE
os.environ['KSPEC_FIBER_EXCLUDE'] = ','.join(BROKEN_FIBERS)
os.environ['KSPEC_FIBER_FIXED']   = ','.join(FIXED_FIBERS)
os.environ['KSPEC_SKY_FIBERS']    = ','.join(SKY_FIBERS)
os.environ['KSPEC_ANGLE_SIGN']    = f'{ANGLE_SIGN[0]},{ANGLE_SIGN[1]}'
os.environ['KSPEC_USE_MEASURED']  = '1'
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, PKG_DIR)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 80
from kspec_metrology import lab
from kspec_metrology.analysis import fiber_config as cfg
print(f'fiber table {cfg.FIBER_TABLE_PATH}: {len(cfg.fiber_ids())} positioners')
print('broken', cfg.FIBER_EXCLUDE, '| fixed', cfg.FIBER_FIXED, '| sky', cfg.SKY_FIBERS, '| angle sign', cfg.ANGLE_SIGN)


In [ ]:

MAGNIFICATION  = 8.40           # camera -> focal 배율 초기값 (2026-10-17 세팅 기준; 몇 % 안으로 맞으면 됨)
MODEL          = 'full+affine'  # 본 장비. fiducial 30개 기준 leave-one-out 이 가장 좋았다 (README 표). 실험실 광학계는 'radial+affine'
NWINDOW        = 40
NITER_RECENTER = 1
THRESHOLD      = 1e3            # 시작 문턱 [ADU, 배경 포함]; 검출 개수에 맞춰 자동으로 올리고 내린다
MIN_DIST       = 80             # 이보다 가까운 검출은 하나로 합침 [px]
FINDER_OPTS    = {'minarea': 9, 'deblend_cont': 1.0}
N_EXTRA        = 0              # FIXED/SKY 외의 여분 밝은 점 (반사광 등)
SAT_LEVEL      = 65000.
TILE           = 'tile01'



## 1. 목표 파일
셋 중 하나. 배열은 표의 positioner 순서로 73개가 모두 채워진다 (끊어진/고정된 것도 포함; 그것들은 결과에서 NaN).


In [ ]:

TARGET_FILE = os.path.join(OUT_DIR, 'calibration_target.info')
lab.make_target(TARGET_FILE, angles=(-30., 150.))        # 모두 같은 각도 (alpha, beta) [deg]
# lab.make_target(TARGET_FILE, offset=(8., 0.))         # 모두 회전 중심에서 같은 (dx, dy) [mm]
print(open(TARGET_FILE).read()[:300], '...')


In [ ]:

FIBER_IMAGE = '/path/to/fiber.fits'
FID_IMAGES  = []
res = lab.measure(TARGET_FILE, FIBER_IMAGE, FID_IMAGES, out_dir=OUT_DIR, tile='calib', itrial=1,
                  magnification=MAGNIFICATION, model=MODEL, nwindow=NWINDOW, threshold=THRESHOLD,
                  niter_recenter=NITER_RECENTER, sat_level=SAT_LEVEL, n_extra=N_EXTRA,
                  min_dist=MIN_DIST, finder_opts=FINDER_OPTS)
tab = lab.offsets_table(res); err = np.hypot(tab.dx_um, tab.dy_um)
print(f'fiber - target: median {np.nanmedian(err):.1f} um, max {np.nanmax(err):.1f} um')
lab.show_offsets(res['offsets']);
tab.round(3)



## 2. positioner 별 실제 각도
측정 위치를 회전 중심 기준으로 각도로 되돌리면, 각 positioner 가 명령 각도에서 얼마나 벗어나 있는지(개체별 zero offset)가 나온다.


In [ ]:

from kspec_metrology.analysis.mtlcal import load_configuration
xo, yo, x, y, fid_flag, nfib = load_configuration(res['target_file'], apply_zenith_offset=False)
a1, a2 = cfg.arm_lengths()
X = x[:nfib] + tab.dx_um.values*1e-3; Y = y[:nfib] + tab.dy_um.values*1e-3
alpha, beta = lab.xy2ab(xo[:nfib], yo[:nfib], X, Y, a1, a2)
cmd_a, cmd_b, _ = lab.read_targets(TARGET_FILE)
A = pd.DataFrame(dict(alpha_meas=alpha, beta_meas=beta, dalpha=((alpha - cmd_a + 180) % 360) - 180, dbeta=beta - cmd_b,
                      r_mm=np.hypot(X - xo[:nfib], Y - yo[:nfib])), index=tab.index)
print(f"alpha offset median {A.dalpha.median():.1f} deg (10-90%: {np.nanpercentile(A.dalpha, 10):.1f}..{np.nanpercentile(A.dalpha, 90):.1f}), "
      f"beta offset median {A.dbeta.median():.1f} deg")
A.round(2)


In [ ]:

info = lab.show_steps(res)
